# Pre-label `dataset_enemy_icons` (`img####.png` only)
Loads the latest model checkpoint and prints top-1 prediction + probability.
This notebook does not rename or modify files.

In [1]:
from pathlib import Path
import re
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

In [2]:
PROJECT_ROOT = Path("../")
CKPT_DIR = PROJECT_ROOT / "checkpoints"
DATASET_DIR = PROJECT_ROOT / "dataset_enemy_icons"

IMG_RE = re.compile(r"^img\d{4}\.png$", re.IGNORECASE)

ckpts = sorted(CKPT_DIR.glob("unit_resnet18_*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    ckpts = sorted(CKPT_DIR.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
ckpt_path = ckpts[0]
print("Using checkpoint:", ckpt_path)

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
assert img_paths, f"No img####.png files found in {DATASET_DIR}"
print(f"Found {len(img_paths)} target files")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
payload = torch.load(ckpt_path, map_location=device)
class_names = payload["class_names"]
image_size = int(payload.get("image_size", 128))
mean = np.array(payload.get("normalize_mean", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.array(payload.get("normalize_std", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.where(std == 0, 1.0, std)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(payload["model_state_dict"], strict=True)
model = model.to(device)
model.eval()

def preprocess_icon_bgr(img):
    resized = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - mean) / std
    chw = np.transpose(rgb, (2, 0, 1))
    return torch.from_numpy(chw).unsqueeze(0).float().to(device)

for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"{p.name} | READ_FAIL | prob=0.0000")
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    print(f"{p.name} | {class_names[int(i)]} | prob={float(v):.4f}")


Using checkpoint: ..\checkpoints\unit_resnet18_20260317_171155.pt
Found 172 target files
img0001.png | Pirate_Ship | prob=0.8415
img0002.png | Flame_Frog | prob=0.8156
img0003.png | Burning_Something | prob=0.9718
img0004.png | ChronoZelius | prob=0.9199
img0005.png | Egypt_Ashura | prob=0.8773
img0006.png | The_Reaper_of_Life | prob=0.5794
img0007.png | Goblin_Mage | prob=0.9325
img0008.png | Charred_Lizard | prob=0.8737
img0009.png | Demonblade | prob=0.9060
img0010.png | Goblin_Mage | prob=0.8990
img0011.png | Gator_Crocodile | prob=0.9498
img0012.png | Burning_Rock | prob=0.7864
img0013.png | Green_Slime | prob=0.8542
img0014.png | Infernal_Dragon | prob=0.9050
img0015.png | Pelican_EXP | prob=0.8745
img0016.png | Skull_Wisp | prob=0.6498
img0017.png | Ignisia | prob=0.6997
img0018.png | Nuu | prob=0.8937
img0019.png | Boss_Goblin | prob=0.7807
img0020.png | Primitive_Butterfly | prob=0.8673
img0021.png | Resident_A | prob=0.9154
img0022.png | Dark_Mage | prob=0.7825
img0023.png | 

In [3]:
ACCEPT_THRESHOLD = 0.40

def next_labeled_name(folder: Path, class_name: str):
    pat = re.compile(rf"^{re.escape(class_name)}_(\\d+)\\.png$", re.IGNORECASE)
    max_idx = 0
    for p in folder.iterdir():
        if not p.is_file():
            continue
        m = pat.match(p.name)
        if m:
            max_idx = max(max_idx, int(m.group(1)))

    idx = max_idx + 1
    while True:
        cand = f"{class_name}_{idx:04d}.png"
        if not (folder / cand).exists():
            return cand
        idx += 1

renamed = 0
kept = 0

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"KEEP {p.name} | READ_FAIL")
        kept += 1
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    prob = float(v)
    pred_name = class_names[int(i)]

    if prob >= ACCEPT_THRESHOLD:
        new_name = next_labeled_name(DATASET_DIR, pred_name)
        new_path = DATASET_DIR / new_name
        p.rename(new_path)
        print(f"RENAME {p.name} -> {new_name} | prob={prob:.4f}")
        renamed += 1
    else:
        print(f"KEEP {p.name} | pred={pred_name} | prob={prob:.4f}")
        kept += 1

print(f"done | renamed={renamed} kept={kept}")


RENAME img0001.png -> Pirate_Ship_0014.png | prob=0.8415
RENAME img0002.png -> Flame_Frog_0017.png | prob=0.8156
RENAME img0003.png -> Burning_Something_0012.png | prob=0.9718
RENAME img0004.png -> ChronoZelius_0015.png | prob=0.9199
RENAME img0005.png -> Egypt_Ashura_0013.png | prob=0.8773
RENAME img0006.png -> The_Reaper_of_Life_0018.png | prob=0.5794
RENAME img0007.png -> Goblin_Mage_0013.png | prob=0.9325
RENAME img0008.png -> Charred_Lizard_0013.png | prob=0.8737
RENAME img0009.png -> Demonblade_0009.png | prob=0.9060
RENAME img0010.png -> Goblin_Mage_0014.png | prob=0.8990
RENAME img0011.png -> Gator_Crocodile_0017.png | prob=0.9498
RENAME img0012.png -> Burning_Rock_0016.png | prob=0.7864
RENAME img0013.png -> Green_Slime_0014.png | prob=0.8542
RENAME img0014.png -> Infernal_Dragon_0014.png | prob=0.9050
RENAME img0015.png -> Pelican_EXP_0012.png | prob=0.8745
RENAME img0016.png -> Skull_Wisp_0017.png | prob=0.6498
RENAME img0017.png -> Ignisia_0017.png | prob=0.6997
RENAME img0